# G0 pre-flight gate — GSM8K + GSM-IC (before building guidance)
Runs unguided CoT, checks every step post-hoc, and reports (a) step-pass rate and (b) error
attributability per bucket, against the pre-committed thresholds. See
`docs/gsm8k_motivation_plan.md`. Run the cells in order; ~40 short generations, single session.

## Cell 1 — Setup (clone gsm8k branch, deps)

In [ ]:

SELF_REPO   = "https://github.com/knx4dmn/dag-motivation-exp"
SELF_BRANCH = "gsm8k"
import subprocess, sys, os, torch
torch_ver = torch.__version__.split("+")[0]
open("/content/constraints.txt", "w").write(f"torch=={torch_ver}\n")
if not os.path.isdir("/content/dag-motivation-exp"):
    subprocess.run(["git", "clone", "-b", SELF_BRANCH, SELF_REPO, "/content/dag-motivation-exp"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-c", "/content/constraints.txt",
                "-e", "/content/dag-motivation-exp[colab]"], check=True)
from huggingface_hub import login; login()
print("torch:", torch.__version__, "| CUDA:", torch.version.cuda)

## Cell 2 — Load model + tokenizer (Llama-3.2-3B, fp16)

In [ ]:

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from motivation_exp import config as C
print("GPU:", torch.cuda.get_device_name(0))
tok = AutoTokenizer.from_pretrained(C.PRIMARY_MODEL)
model = AutoModelForCausalLM.from_pretrained(C.PRIMARY_MODEL, torch_dtype=torch.float16,
                                             attn_implementation="sdpa", device_map="cuda").eval()
print("loaded", C.PRIMARY_MODEL)

## Cell 3 — G0 data: load GSM8K test, bucket ~20 items to 1k/2k
Seeded template distractors; validation gate; the same bucketing interface as the full pipeline.

In [ ]:

from motivation_exp import gsm8k_datagen as dg
N_ITEMS = 22          # a few spare for validation drops
G0_BUCKETS = [1024, 2048]
count_tokens = dg.hf_token_counter(tok)
bases = dg.load_gsm8k(split="test", limit=N_ITEMS)     # GSM8K TEST split (amendment C)
buckets = dg.bucket_gsm_to_targets(bases, count_tokens, G0_BUCKETS, base_seed=0, log=print)
items = []
for b in G0_BUCKETS:
    kept = [it for it in buckets[b] if dg.validate_gsm_item(it, count_tokens)[0]][:20]
    items += kept
    print(f"bucket {b}: {len(kept)} valid items, mean tokens "
          f"{sum(i.token_count for i in kept)/max(1,len(kept)):.0f}")

## Cell 4 — Run G0 + verdict
Unguided greedy CoT, post-hoc MathChecker on every step. Prints the per-bucket table and the
PROCEED/STOP verdict against the pre-committed thresholds ((a) step-pass >= 0.50 in both, (b)
catchable >= 1/3). Saves per-item/per-step verdicts for failure analysis.

In [ ]:

import json
from motivation_exp.g0_gsm8k import run_g0, g0_aggregate, print_g0
rows = run_g0(model, tok, items, n_per_bucket=20)
with open("/content/g0_verdicts.jsonl", "w") as f:
    for r in rows: f.write(json.dumps(r) + "\n")
print_g0(g0_aggregate(rows))
# peek a few checker-invisible wrong items (clean arithmetic, wrong plan) for the failure log
print("\nsample checker-invisible wrong items:")
for r in [r for r in rows if not r["correct"] and not r["has_reject"]][:3]:
    print(f"  [{r['item_id']} b{r['bucket']}] pred={r['pred']} gold={r['gold']}")
    for v in r["verdicts"][:4]: print(f"     {v['accepted']}  {v['step']!r}")